In [0]:
from pyspark.sql import functions as sf
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# Read silver layer data
df_silver = spark.read.format("delta").load("/Volumes/weather/default/02_silver")

df_silver.printSchema()

In [0]:
# Daily aggregations by city
df_daily = df_silver.withColumn(
    "observation_date", sf.to_date(sf.col("observation_timestamp_utc"))
)

# Calculate daily statistics by city
df_daily_agg = df_daily.groupBy("city", "country", "observation_date").agg(
    # Temperature metrics
    sf.avg("temp_c").alias("avg_temp_c"),
    sf.min("temp_min_c").alias("min_temp_c"),
    sf.max("temp_max_c").alias("max_temp_c"),
    sf.avg("feels_like_c").alias("avg_feels_like_c"),
    
    # Atmospheric metrics
    sf.avg("pressure_hpa").alias("avg_pressure_hpa"),
    sf.avg("humidity_pct").alias("avg_humidity_pct"),
    
    # Wind metrics
    sf.avg("wind_ms").alias("avg_wind_ms"),
    sf.max("wind_ms").alias("max_wind_ms"),
    sf.avg("wind_gust_ms").alias("avg_wind_gust_ms"),
    sf.max("wind_gust_ms").alias("max_wind_gust_ms"),
    
    # Cloud and rain metrics
    sf.avg("clouds_pct").alias("avg_clouds_pct"),
    sf.sum("rain_mm_h").alias("total_rain_mm"),
    
    # Most common weather condition
    sf.first("weather_main").alias("dominant_weather"),
    
    # Observation counts
    sf.count("*").alias("observation_count"),
    
    # Coordinates (first value)
    sf.first("coord_lat").alias("coord_lat"),
    sf.first("coord_lon").alias("coord_lon")
).orderBy("city", "observation_date")

# Add derived metrics
df_daily_agg = df_daily_agg.withColumn(
    "temp_range_c", sf.col("max_temp_c") - sf.col("min_temp_c")
)

df_daily_agg = df_daily_agg.withColumn(
    "ingestion_timestamp", sf.current_timestamp()
)

display(df_daily_agg)

In [0]:
# Save daily aggregations to gold layer making sure there are no duplicate records
table_name = "weather.default.03_gold_daily"

if not spark.catalog.tableExists(table_name):
    df_daily_agg.write.format("delta").saveAsTable(table_name)

else:
    delta_table = DeltaTable.forName(spark, table_name)
    delta_table.alias("target").merge(
        df_daily_agg.alias("source"),
        """
        target.city = source.city
        AND target.observation_date = source.observation_date
        AND target.country = source.country
        """,
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
# Weather pattern summary - most common conditions by city
df_weather_summary = df_silver.groupBy("city", "country", "weather_main").agg(
    sf.count("*").alias("occurrence_count"),
    sf.round(sf.avg("temp_c"), 2).alias("avg_temp_during_condition_c"),
    sf.avg("humidity_pct").alias("avg_humidity_during_condition_pct"),
    sf.first("coord_lat").alias("coord_lat"),
    sf.first("coord_lon").alias("coord_lon")
)

# Calculate percentage of time each weather condition occurs
window_spec = Window.partitionBy("city", "country")
df_weather_summary = df_weather_summary.withColumn(
    "total_observations", sf.sum("occurrence_count").over(window_spec)
)

df_weather_summary = df_weather_summary.withColumn(
    "occurrence_pct", 
    sf.round((sf.col("occurrence_count") / sf.col("total_observations")) * 100, 2)
)

df_weather_summary = df_weather_summary.withColumn(
    "ingestion_timestamp", sf.current_timestamp()
)

df_weather_summary = df_weather_summary.orderBy(
    "city", sf.desc("occurrence_count")
)

display(df_weather_summary)

In [0]:
# Save weather pattern summary to gold layer
df_weather_summary.write.mode("overwrite").saveAsTable(
    "weather.default.03_gold_summary"
)


In [0]:
# Hourly aggregations by city
df_hourly = df_silver.withColumn(
    "observation_hour", sf.date_trunc("hour", sf.col("observation_timestamp_utc"))
)

# Calculate hourly statistics by city
df_hourly_agg = df_hourly.groupBy("city", "country", "observation_hour").agg(
    # Temperature metrics
    sf.avg("temp_c").alias("avg_temp_c"),
    sf.min("temp_c").alias("min_temp_c"),
    sf.max("temp_c").alias("max_temp_c"),
    sf.avg("feels_like_c").alias("avg_feels_like_c"),
    
    # Atmospheric metrics
    sf.avg("pressure_hpa").alias("avg_pressure_hpa"),
    sf.avg("humidity_pct").alias("avg_humidity_pct"),
    
    # Wind metrics
    sf.avg("wind_ms").alias("avg_wind_ms"),
    sf.max("wind_ms").alias("max_wind_ms"),
    sf.avg("wind_deg").alias("avg_wind_deg"),
    
    # Cloud and rain metrics
    sf.avg("clouds_pct").alias("avg_clouds_pct"),
    sf.sum("rain_mm_h").alias("total_rain_mm"),
    
    # Weather condition
    sf.first("weather_main").alias("weather_condition"),
    sf.first("weather_desc").alias("weather_description"),
    
    # Observation counts
    sf.count("*").alias("observation_count"),
    
    # Coordinates
    sf.first("coord_lat").alias("coord_lat"),
    sf.first("coord_lon").alias("coord_lon")
).orderBy("city", "observation_hour")

df_hourly_agg = df_hourly_agg.withColumn(
    "ingestion_timestamp", sf.current_timestamp()
)

display(df_hourly_agg)

In [0]:
# Save hourly aggregations to gold layer
df_hourly_agg.write.mode("append").saveAsTable(
    "weather.default.03_gold_hourly"
)


In [0]:
# Top temperatures by city
df_top_temps = df_silver.groupBy("city").agg(sf.max("temp_c").alias("max_temp_c"))

# Save 5 top temperatures as table
df_top_temps.orderBy(sf.desc("max_temp_c")).limit(5).write.mode(
    "overwrite"
).saveAsTable("weather.default.03_gold_top_temps")